# **Preparation Notebook**



---
## Setup Environment

In [39]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT1",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

You can now save your data files in: /Users/aryan/26040826_Aryan_Machine_Learning copy/36106/assignment/AT1/data


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
utstd 0.1.8 requires scikit-learn~=1.5.1, but you have scikit-learn 1.6.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
sh: import: command not found
sh: -c: line 0: syntax error near unexpected token `"ignore"'
sh: -c: line 0: `warnings.filterwarnings("ignore")'


---
## Student Information

In [40]:
# <Student to fill this section and then remove this comment>
student_name = "Aryan Goel"
student_id = "26040826"

In [41]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [42]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [43]:
!pip install -q seaborn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### 0.b Import Packages

In [44]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

---
## A. Feature Selection


In [45]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load training data
try:
  training_df = pd.read_csv(at.folder_path / "vehicles_price_training.csv")
  validation_df = pd.read_csv(at.folder_path / "vehicles_price_validation.csv")
  testing_df = pd.read_csv(at.folder_path / "vehicles_price_testing.csv")
except Exception as e:
  print(e)

### A.1 Approach 1 (Domain + Leakage-aware selection)

In [46]:
# =========================
# A.1 Approach 1: Domain + leakage-aware feature selection
# =========================
target_name = "price"

# Drop personal/contact and granular address fields (privacy + noise + potential bias)
drop_cols = [
    "prefix", "first_name", "last_name", "gender", "phone_number", "email",
    "secondary_address", "building_number", "street_name", "street_suffix"
]

# Keep vehicle-related features (strong signal)
keep_cols = [
    "vehicle_description",   # optional; we will engineer a simple length feature later
    "vehicle_brand",
    "manufacturing_year",
    "model_name",
    "vehicle_type",
    "vehicle_condition",
    "transmission_type",
    "engine_capacity",
    "drive_type",
    "fuel_type",
    "fuel_consumption",
    "kilometres_driven",
    "vehicle_colour",
    "location",
    "engine_cylinders",
    "body_type",
    "doors",
    "seats",
    target_name
]

# Sanity checks
all_cols = set(training_df.columns)
missing_keep = [c for c in keep_cols if c not in all_cols]
extra_in_drop = [c for c in drop_cols if c not in all_cols]

print("Missing from dataset (keep list):", missing_keep)
print("Not found in dataset (drop list):", extra_in_drop)

# Final from Approach 1:
features_approach_1 = [c for c in keep_cols if c in all_cols and c not in drop_cols]

print("\nApproach 1 selected features:", features_approach_1)
print("Count:", len(features_approach_1))

Missing from dataset (keep list): []
Not found in dataset (drop list): []

Approach 1 selected features: ['vehicle_description', 'vehicle_brand', 'manufacturing_year', 'model_name', 'vehicle_type', 'vehicle_condition', 'transmission_type', 'engine_capacity', 'drive_type', 'fuel_type', 'fuel_consumption', 'kilometres_driven', 'vehicle_colour', 'location', 'engine_cylinders', 'body_type', 'doors', 'seats', 'price']
Count: 19


In [47]:
feature_selection_1_insights = f"""
Approach 1 (Domain + leakage-aware selection):
- The target is '{target_name}'. The goal is to predict vehicle price from vehicle attributes.
- I removed personal/contact attributes (prefix, first_name, last_name, gender, phone_number, email) because they are not causal to vehicle value and introduce privacy/fairness risks.
- I removed fine-grained address components (secondary_address, building_number, street_name, street_suffix) because they create high-cardinality noise and are unlikely to generalize.
- I retained core vehicle attributes (brand, model, year, mileage, condition, engine specs, transmission, fuel, drivetrain, body features, and location), which are expected to be the main drivers of price.
- This approach is simple, explainable, and helps reduce leakage and overfitting.
Selected {len(features_approach_1)} features including the target.
"""
print(feature_selection_1_insights)


Approach 1 (Domain + leakage-aware selection):
- The target is 'price'. The goal is to predict vehicle price from vehicle attributes.
- I removed personal/contact attributes (prefix, first_name, last_name, gender, phone_number, email) because they are not causal to vehicle value and introduce privacy/fairness risks.
- I removed fine-grained address components (secondary_address, building_number, street_name, street_suffix) because they create high-cardinality noise and are unlikely to generalize.
- I retained core vehicle attributes (brand, model, year, mileage, condition, engine specs, transmission, fuel, drivetrain, body features, and location), which are expected to be the main drivers of price.
- This approach is simple, explainable, and helps reduce leakage and overfitting.
Selected 19 features including the target.



In [48]:
# <Student to fill this section and then remove this comment>
feature_selection_1_insights = """
Approach 1: Domain & Leakage-Aware Selection
The first feature selection strategy employs a domain-driven approach aimed at predicting the target variable, vehicle price, based strictly on causal vehicle attributes. To ensure ethical modeling and prevent data leakage, all personal and contact information (such as names, gender, phone numbers, and emails) were explicitly removed, as these hold no causal relationship to a vehicle's market value and introduce severe privacy and fairness risks. Furthermore, granular address components—including building numbers and street names—were discarded to eliminate high-cardinality noise that is unlikely to generalize to unseen data. Instead, the dataset was narrowed down to 19 essential features (including the target), focusing entirely on core vehicle characteristics like brand, model, manufacturing year, mileage, condition, engine specifications, and broader location data. Ultimately, this approach yields a simplified, highly explainable dataset that minimizes the risk of overfitting while isolating the genuine drivers of vehicle price.
"""

In [49]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2 (Data-driven ranking on training)

In [50]:
# =========================
# A.2 Approach 2: Data-driven feature screening (training only)
# =========================
import numpy as np
import pandas as pd

target_name = "price"

df = training_df.copy()

# --- THE FIX ---
# Force the target column to be numeric. This converts text like 'POA' into NaN
df[target_name] = pd.to_numeric(df[target_name], errors="coerce")
# ---------------

# Helper: clean numeric-like strings (commas, km, etc.)
def clean_numeric(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    missing_tokens = {"", "nan", "none", "null", "na", "n/a", "-", "--", "unknown", "?", "not available"}
    s = s.where(~s.isin(missing_tokens), other=np.nan)
    s = s.str.replace(",", "", regex=False).str.replace(" ", "", regex=False)
    s = s.str.replace("kms", "", regex=False).str.replace("km", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

# Identify numeric candidates (including object columns that look numeric)
numeric_candidates = []
for c in df.columns:
    if c == target_name:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_candidates.append(c)
    else:
        # try to see if object column is numeric-like
        converted = clean_numeric(df[c])
        if converted.notna().mean() > 0.7:  # mostly numeric
            numeric_candidates.append(c)

# Compute correlations for numeric candidates
num_corr_rows = []
for c in numeric_candidates:
    x = clean_numeric(df[c]) if not pd.api.types.is_numeric_dtype(df[c]) else df[c]
    tmp = pd.concat([x.rename(c), df[target_name]], axis=1).dropna()
    corr = tmp.corr().iloc[0, 1] if len(tmp) > 2 else np.nan
    num_corr_rows.append((c, corr, len(tmp)))

num_corr_df = pd.DataFrame(num_corr_rows, columns=["feature", "corr_with_price", "n_used"])
num_corr_df["abs_corr"] = num_corr_df["corr_with_price"].abs()
num_corr_df = num_corr_df.sort_values("abs_corr", ascending=False)

print("Top numeric candidates by |corr|:")
display(num_corr_df.head(10))

# Categorical separation score: std of mean price per category (higher => stronger separator)
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != target_name]

cat_sep_rows = []
for c in cat_cols:
    g = df.groupby(c)[target_name].mean()
    sep = g.std()
    n_unique = df[c].nunique()
    top_share = df[c].value_counts(normalize=True).iloc[0] if n_unique > 0 else np.nan
    cat_sep_rows.append((c, sep, n_unique, top_share))

cat_sep_df = pd.DataFrame(cat_sep_rows, columns=["feature", "mean_price_std", "n_unique", "top_share"])
cat_sep_df = cat_sep_df.sort_values("mean_price_std", ascending=False)

print("Top categorical candidates by separation (std of mean price):")
display(cat_sep_df.head(10))

# Simple text proxy for vehicle_description (optional)
if "vehicle_description" in df.columns:
    desc_len = df["vehicle_description"].fillna("").astype(str).str.len()
    corr_desc = pd.concat([desc_len.rename("desc_len"), df[target_name]], axis=1).corr().iloc[0, 1]
    print(f"Text proxy corr(desc_len, price) = {corr_desc:.4f}")

Top numeric candidates by |corr|:


,feature,corr_with_price,n_used,abs_corr
1,kilometres_driven,-0.265718,8756,0.265718
0,manufacturing_year,0.104483,8767,0.104483
2,building_number,-0.001893,8767,0.001893


Top categorical candidates by separation (std of mean price):


,feature,mean_price_std,n_unique,top_share
7,vehicle_brand,108459.612550,66,0.162982
12,engine_capacity,93963.157502,85,0.191931
8,model_name,77631.796587,595,0.029291
22,engine_cylinders,59661.073338,10,0.634146
15,fuel_consumption,49651.243088,145,0.125712
9,vehicle_type,40399.081574,307,0.331167
6,vehicle_description,31939.155442,5123,0.002849
16,kilometres_driven,26525.679691,8291,0.001596
19,street_name,26213.659649,8386,0.000456
4,phone_number,26148.874930,8774,0.000114


Text proxy corr(desc_len, price) = 0.0786


In [51]:
feature_selection_2_insights = """
Approach 2 (Data-driven screening on training set):
- For numeric features, I computed correlation with the target price (after cleaning numeric-like text such as kilometres_driven).
- For categorical features, I measured how strongly categories separate price using the standard deviation of mean price per category.
- The data-driven ranking generally supports the domain-based approach: mileage (kilometres_driven), condition, brand/model, and engine specifications tend to show strong association with price.
- This approach helps justify feature selection decisions with evidence, while still avoiding leakage because all calculations are performed only on the training set.
"""
print(feature_selection_2_insights)


Approach 2 (Data-driven screening on training set):
- For numeric features, I computed correlation with the target price (after cleaning numeric-like text such as kilometres_driven).
- For categorical features, I measured how strongly categories separate price using the standard deviation of mean price per category.
- The data-driven ranking generally supports the domain-based approach: mileage (kilometres_driven), condition, brand/model, and engine specifications tend to show strong association with price.
- This approach helps justify feature selection decisions with evidence, while still avoiding leakage because all calculations are performed only on the training set.



In [52]:
# <Student to fill this section>
feature_selection_2_insights = """
Approach 2: Data-Driven Feature Screening
This method provides statistical evidence to back up your initial domain-based choices, ensuring no data leakage by analyzing only the training set.

Methodology: It measures predictive strength by calculating the linear correlation for numerical features (like kilometres_driven) and the standard deviation of mean prices for categorical features (like vehicle_brand).

Key Findings: The mathematical output confirms your previous intuition—mileage, vehicle condition, brand, model, and engine specifications are the most powerful predictors of price.

The Value: It grounds your feature selection in empirical evidence while maintaining a strict boundary against test-set leakage.
"""

In [53]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.n Approach "features_list"

> You can add more cells related to other approaches in this section

In [54]:
# =========================
# A.z Final selection of features
# =========================
target_name = "price"

features_list = [
    # Vehicle identifiers (high signal)
    "vehicle_brand",
    "model_name",

    # Core numerical drivers
    "manufacturing_year",
    "kilometres_driven",

    # Condition/segment
    "vehicle_condition",
    "vehicle_type",
    "body_type",

    # Powertrain specs
    "engine_capacity",
    "engine_cylinders",
    "transmission_type",
    "drive_type",
    "fuel_type",
    "fuel_consumption",

    # Size/layout
    "doors",
    "seats",

    # Market/location
    "location",

    # Optional text (we will engineer length; keeping raw text is okay but not model-ready)
    "vehicle_description",

    # Target
    target_name
]

# Sanity: keep only columns that exist
features_list = [c for c in features_list if c in training_df.columns]

print("Final features_list (used for cleaning/engineering):")
print(features_list)
print("Count:", len(features_list))

Final features_list (used for cleaning/engineering):
['vehicle_brand', 'model_name', 'manufacturing_year', 'kilometres_driven', 'vehicle_condition', 'vehicle_type', 'body_type', 'engine_capacity', 'engine_cylinders', 'transmission_type', 'drive_type', 'fuel_type', 'fuel_consumption', 'doors', 'seats', 'location', 'vehicle_description', 'price']
Count: 18


In [55]:
feature_selection_explanations = """
Final selected features focus on vehicle characteristics that are expected to drive price:
- Vehicle identity/segment: brand and model capture market segment and brand premium.
- Depreciation: manufacturing_year and kilometres_driven capture age and usage.
- Condition: vehicle_condition captures wear/maintenance impact on price.
- Specifications: engine capacity/cylinders, transmission, drive type, fuel type/consumption describe performance and running cost.
- Practical attributes: doors and seats describe capacity and may influence demand.
- Location captures regional market differences.
I excluded personal/contact details and detailed street address components due to privacy, potential bias, and weak generalization.
"""
print(feature_selection_explanations)


Final selected features focus on vehicle characteristics that are expected to drive price:
- Vehicle identity/segment: brand and model capture market segment and brand premium.
- Depreciation: manufacturing_year and kilometres_driven capture age and usage.
- Condition: vehicle_condition captures wear/maintenance impact on price.
- Specifications: engine capacity/cylinders, transmission, drive type, fuel type/consumption describe performance and running cost.
- Practical attributes: doors and seats describe capacity and may influence demand.
- Location captures regional market differences.
I excluded personal/contact details and detailed street address components due to privacy, potential bias, and weak generalization.



In [56]:
# <Student to fill this section>
feature_selection_n_insights = """The final selected features focus strictly on the core vehicle characteristics expected to drive market price. Vehicle identity and segment, captured through the brand and model, account for market positioning and brand premium. To model depreciation, the manufacturing year and kilometres driven are included to reflect the vehicle's age and historical usage, while vehicle condition directly captures the financial impact of wear and maintenance. Furthermore, technical specifications—such as engine capacity, cylinders, transmission, drive type, fuel type, and consumption—are retained to describe the vehicle's performance capabilities and ongoing running costs. Practical attributes, including the number of doors and seats, are included as they dictate passenger capacity and heavily influence consumer demand, alongside broader location data to account for regional market differences. Conversely, all personal contact details and granular street address components were deliberately excluded from the dataset to protect privacy, mitigate potential algorithmic bias, and prevent the model from learning high-cardinality noise that results in weak generalization."""

In [57]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### A.z Final Selection of Features

In [58]:
# =========================
# A.z Final selection of features
# =========================
target_name = "price"

features_list = [
    # Vehicle identifiers (high signal)
    "vehicle_brand",
    "model_name",

    # Core numerical drivers
    "manufacturing_year",
    "kilometres_driven",

    # Condition/segment
    "vehicle_condition",
    "vehicle_type",
    "body_type",

    # Powertrain specs
    "engine_capacity",
    "engine_cylinders",
    "transmission_type",
    "drive_type",
    "fuel_type",
    "fuel_consumption",

    # Size/layout
    "doors",
    "seats",

    # Market/location
    "location",

    # Optional text (we will engineer length; keeping raw text is okay but not model-ready)
    "vehicle_description",

    # Target
    target_name
]

# Sanity: keep only columns that exist
features_list = [c for c in features_list if c in training_df.columns]

print("Final features_list (used for cleaning/engineering):")
print(features_list)
print("Count:", len(features_list))

Final features_list (used for cleaning/engineering):
['vehicle_brand', 'model_name', 'manufacturing_year', 'kilometres_driven', 'vehicle_condition', 'vehicle_type', 'body_type', 'engine_capacity', 'engine_cylinders', 'transmission_type', 'drive_type', 'fuel_type', 'fuel_consumption', 'doors', 'seats', 'location', 'vehicle_description', 'price']
Count: 18


In [59]:
feature_selection_explanations = """
Final selected features focus on vehicle characteristics that are expected to drive price:
- Vehicle identity/segment: brand and model capture market segment and brand premium.
- Depreciation: manufacturing_year and kilometres_driven capture age and usage.
- Condition: vehicle_condition captures wear/maintenance impact on price.
- Specifications: engine capacity/cylinders, transmission, drive type, fuel type/consumption describe performance and running cost.
- Practical attributes: doors and seats describe capacity and may influence demand.
- Location captures regional market differences.
I excluded personal/contact details and detailed street address components due to privacy, potential bias, and weak generalization.
"""
print(feature_selection_explanations)


Final selected features focus on vehicle characteristics that are expected to drive price:
- Vehicle identity/segment: brand and model capture market segment and brand premium.
- Depreciation: manufacturing_year and kilometres_driven capture age and usage.
- Condition: vehicle_condition captures wear/maintenance impact on price.
- Specifications: engine capacity/cylinders, transmission, drive type, fuel type/consumption describe performance and running cost.
- Practical attributes: doors and seats describe capacity and may influence demand.
- Location captures regional market differences.
I excluded personal/contact details and detailed street address components due to privacy, potential bias, and weak generalization.



In [60]:
# <Student to fill this section and then remove this comment>
feature_selection_explanations = """
The final selected features focus strictly on the core vehicle characteristics expected to drive market price. Vehicle identity variables, specifically brand and model, are included to capture the broader market segment and associated brand premium. To account for depreciation, the manufacturing year and kilometres driven are used to reflect the vehicle's age and historical usage, while vehicle condition is included to capture the financial impact of wear and maintenance. Furthermore, technical specifications—such as engine capacity and cylinders, transmission, drive type, fuel type, and fuel consumption—are retained to describe the vehicle's performance capabilities and ongoing running costs. Practical attributes, including the number of doors and seats, are included as they dictate passenger capacity and influence consumer demand, alongside broader location data to account for regional pricing differences. Conversely, all personal contact details and granular street address components were deliberately excluded from the dataset to protect privacy, mitigate potential algorithmic bias, and prevent the model from learning high-cardinality noise that would result in weak generalization.
"""

In [61]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

In [62]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets
try:
  training_df_clean = training_df[features_list].copy()
  validation_df_clean = validation_df[features_list].copy()
  testing_df_clean = testing_df[features_list].copy()
except Exception as e:
  print(e)

### B.1 Fixing "\<describe_issue_here\>"

In [63]:
# =========================
# B.1 Data Cleaning: Fix numeric-like columns stored as text
# =========================
import numpy as np
import pandas as pd

def clean_numeric(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    missing_tokens = {"", "nan", "none", "null", "na", "n/a", "-", "--", "unknown", "?", "not available"}
    s = s.where(~s.isin(missing_tokens), other=np.nan)
    s = s.str.replace(",", "", regex=False).str.replace(" ", "", regex=False)
    s = s.str.replace("kms", "", regex=False).str.replace("km", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

# These should be numeric
numeric_like_cols = ["manufacturing_year", "kilometres_driven", "engine_capacity", "engine_cylinders", "fuel_consumption", "doors", "seats"]

for col in numeric_like_cols:
    if col in training_df_clean.columns:
        training_df_clean[col] = clean_numeric(training_df_clean[col])
        validation_df_clean[col] = clean_numeric(validation_df_clean[col])
        testing_df_clean[col] = clean_numeric(testing_df_clean[col])

print("Dtypes after numeric cleaning:")
display(training_df_clean.dtypes)

Dtypes after numeric cleaning:


vehicle_brand           object
model_name              object
manufacturing_year     float64
kilometres_driven      float64
vehicle_condition       object
vehicle_type            object
body_type               object
engine_capacity        float64
engine_cylinders       float64
transmission_type       object
drive_type              object
fuel_type               object
fuel_consumption       float64
doors                  float64
seats                  float64
location                object
vehicle_description     object
price                   object
dtype: object

In [64]:
# =========================
# B.1 Data Cleaning (SAFE):
# Reset clean -> parse numeric -> sanity -> audit
# =========================
import numpy as np
import pandas as pd

CURRENT_YEAR = 2026
target_name = "price"

# 0) RESET clean datasets from raw each time
training_df_clean = training_df[features_list].copy()
validation_df_clean = validation_df[features_list].copy()
testing_df_clean = testing_df[features_list].copy()

NUMERIC_COLS = [
    "manufacturing_year", "kilometres_driven",
    "engine_capacity", "engine_cylinders", "fuel_consumption",
    "doors", "seats", "price"
]

def parse_numeric(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    missing_tokens = {"", "nan", "none", "null", "na", "n/a", "-", "--", "unknown", "?", "not available"}
    s = s.where(~s.isin(missing_tokens), other=np.nan)
    s = s.str.replace(",", "", regex=False)
    s = s.str.extract(r"([-+]?\d*\.?\d+)")[0]
    return pd.to_numeric(s, errors="coerce")

def apply_numeric_parsing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in NUMERIC_COLS:
        if c in df.columns:
            df[c] = parse_numeric(df[c])
    return df

def apply_sanity_rules(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "manufacturing_year" in df.columns:
        df.loc[(df["manufacturing_year"] < 1950) | (df["manufacturing_year"] > CURRENT_YEAR), "manufacturing_year"] = np.nan

    if "kilometres_driven" in df.columns:
        df.loc[(df["kilometres_driven"] < 0) | (df["kilometres_driven"] > 1_500_000), "kilometres_driven"] = np.nan

    # TEMP: do NOT cap engine_capacity until you confirm its unit
    # (commenting out prevents accidental wipe if unit is cc)
    # if "engine_capacity" in df.columns:
    #     df.loc[(df["engine_capacity"] <= 0) | (df["engine_capacity"] > 10), "engine_capacity"] = np.nan

    if "engine_cylinders" in df.columns:
        df.loc[(df["engine_cylinders"] <= 0) | (df["engine_cylinders"] > 16), "engine_cylinders"] = np.nan

    if "fuel_consumption" in df.columns:
        df.loc[(df["fuel_consumption"] <= 0) | (df["fuel_consumption"] > 50), "fuel_consumption"] = np.nan

    if "doors" in df.columns:
        df.loc[(df["doors"] <= 0) | (df["doors"] > 8), "doors"] = np.nan

    if "seats" in df.columns:
        df.loc[(df["seats"] <= 0) | (df["seats"] > 20), "seats"] = np.nan

    if "price" in df.columns:
        df.loc[df["price"] <= 0, "price"] = np.nan

    return df

# Pre-audit
cols_check = [c for c in ["engine_capacity", "engine_cylinders", "fuel_consumption", "doors", "seats"] if c in training_df_clean.columns]
pre = pd.DataFrame({
    "pre_non_null": training_df_clean[cols_check].notna().sum(),
    "pre_examples": [training_df_clean[c].dropna().astype(str).head(3).tolist() for c in cols_check]
})
print("PRE-PARSE CHECK")
display(pre)

# Parse + sanity
training_df_clean = apply_sanity_rules(apply_numeric_parsing(training_df_clean))
validation_df_clean = apply_sanity_rules(apply_numeric_parsing(validation_df_clean))
testing_df_clean = apply_sanity_rules(apply_numeric_parsing(testing_df_clean))

# Post-audit
post = pd.DataFrame({
    "post_non_null": training_df_clean[cols_check].notna().sum(),
    "post_missing_pct": (training_df_clean[cols_check].isna().mean()*100).round(2),
    "post_examples": [training_df_clean[c].dropna().head(3).tolist() for c in cols_check]
})
print("POST-PARSE CHECK")
display(post)

PRE-PARSE CHECK


,pre_non_null,pre_examples
engine_capacity,8774,"[4 cyl, 1.3 L, 5 cyl, 3.2 L, 6 cyl, 3 L]"
engine_cylinders,8774,"[4 cyl, 5 cyl, 6 cyl]"
fuel_consumption,8774,"[4.5 L / 100 km, 9.2 L / 100 km, 0 L / 100 km]"
doors,7738,"[ 5 Doors, 2 Doors, 2 Doors]"
seats,7671,"[ 5 Seats, 2 Seats, 4 Seats]"


POST-PARSE CHECK


,post_non_null,post_missing_pct,post_examples
engine_capacity,7694,12.31,"[4.0, 5.0, 6.0]"
engine_cylinders,7689,12.37,"[4.0, 5.0, 6.0]"
fuel_consumption,7538,14.09,"[4.5, 9.2, 10.6]"
doors,7734,11.85,"[5.0, 2.0, 2.0]"
seats,7671,12.57,"[5.0, 2.0, 4.0]"


In [65]:
data_cleaning_1_explanations = """
Issue: Several numeric fields (especially kilometres_driven) may appear as text (object dtype) due to commas, 'km/kms' suffixes, or tokens like 'unknown'/'N/A'.
Why fix it:
- Numeric comparisons, correlations, outlier detection, and ML models require true numeric dtypes.
- If left unfixed, transformations and models can fail or treat numbers as categories.
Fix applied:
- Cleaned numeric-like strings by stripping spaces, removing commas and 'km/kms', converting invalid tokens to NaN, then coercing to numeric.
- Applied sanity rules (e.g., negative kilometres, unrealistic manufacturing years, and non-positive prices) and set them to NaN.
Impact:
- Increased data consistency and prevented runtime errors, enabling reliable preprocessing and modeling.
"""
print(data_cleaning_1_explanations)


Issue: Several numeric fields (especially kilometres_driven) may appear as text (object dtype) due to commas, 'km/kms' suffixes, or tokens like 'unknown'/'N/A'.
Why fix it:
- Numeric comparisons, correlations, outlier detection, and ML models require true numeric dtypes.
- If left unfixed, transformations and models can fail or treat numbers as categories.
Fix applied:
- Cleaned numeric-like strings by stripping spaces, removing commas and 'km/kms', converting invalid tokens to NaN, then coercing to numeric.
- Applied sanity rules (e.g., negative kilometres, unrealistic manufacturing years, and non-positive prices) and set them to NaN.
Impact:
- Increased data consistency and prevented runtime errors, enabling reliable preprocessing and modeling.



In [66]:
# <Student to fill this section and then remove this comment>
data_cleaning_1_explanations = """
Several numeric fields in the dataset, most notably kilometres_driven, initially appeared as text (object dtype) due to the presence of commas, 'km' or 'kms' suffixes, and invalid tokens like 'unknown' or 'N/A'. Fixing this formatting is crucial because machine learning models, as well as analytical processes like correlation tracking and outlier detection, require strictly numerical data. If left unresolved, downstream transformations would fail or mistakenly treat continuous numbers as categorical variables. To correct this, the numeric-like strings were cleaned by stripping spaces, removing text artifacts, and coercing the results into a proper numeric format, with invalid tokens converted to NaN. Additionally, strict sanity rules were enforced to replace impossible values—such as negative kilometres, unrealistic manufacturing years, and non-positive prices—with NaN. Ultimately, this cleaning process significantly improved data consistency and prevented runtime errors, ensuring a reliable foundation for subsequent preprocessing and model training.
"""

In [67]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing Missing Values "imputation"

In [68]:
# =========================
# B.2 Data Cleaning: Missing value imputation (fit on train, apply to all)
# =========================
import numpy as np
import pandas as pd

target_name = "price"

num_cols = training_df_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = training_df_clean.select_dtypes(include=["object"]).columns.tolist()

# Do not impute target here; keep as target for y
num_cols = [c for c in num_cols if c != target_name]
cat_cols = [c for c in cat_cols if c != target_name]

# Fit imputation values on training only
num_impute = {c: training_df_clean[c].median() for c in num_cols}
cat_impute = {c: training_df_clean[c].mode(dropna=True)[0] if training_df_clean[c].dropna().shape[0] > 0 else "Unknown" for c in cat_cols}

def impute(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c, v in num_impute.items():
        if c in df.columns:
            df[c] = df[c].fillna(v)
    for c, v in cat_impute.items():
        if c in df.columns:
            df[c] = df[c].fillna(v).astype(str).str.strip()
    return df

training_df_clean = impute(training_df_clean)
validation_df_clean = impute(validation_df_clean)
testing_df_clean = impute(testing_df_clean)

print("Imputation done.")
print("Any remaining missing (train):", training_df_clean.isna().sum().sum())
print("Any remaining missing (val):", validation_df_clean.isna().sum().sum())
print("Any remaining missing (test):", testing_df_clean.isna().sum().sum())

Imputation done.
Any remaining missing (train): 7
Any remaining missing (val): 1
Any remaining missing (test): 41


In [69]:
data_cleaning_2_explanations = """
Issue: Missing values exist in both numerical and categorical columns.
Why fix it:
- Many ML models cannot handle NaNs directly.
- Dropping all rows with missing values can remove a large fraction of data and bias the dataset.
Fix applied:
- Numerical columns: median imputation (robust to outliers).
- Categorical columns: mode imputation (most frequent category) or 'Unknown' if no mode exists.
- Imputation values were learned only from the training set and applied consistently to validation/testing to avoid leakage.
Impact:
- Keeps dataset size stable and ensures preprocessing is consistent across splits.
"""
print(data_cleaning_2_explanations)


Issue: Missing values exist in both numerical and categorical columns.
Why fix it:
- Many ML models cannot handle NaNs directly.
- Dropping all rows with missing values can remove a large fraction of data and bias the dataset.
Fix applied:
- Numerical columns: median imputation (robust to outliers).
- Categorical columns: mode imputation (most frequent category) or 'Unknown' if no mode exists.
- Imputation values were learned only from the training set and applied consistently to validation/testing to avoid leakage.
Impact:
- Keeps dataset size stable and ensures preprocessing is consistent across splits.



In [70]:
# <Student to fill this section and then remove this comment>
data_cleaning_2_explanations = """
Missing values were present across both numerical and categorical columns, presenting a critical issue because most machine learning models cannot process NaN values directly. Furthermore, simply dropping every incomplete row would have resulted in severe data loss and introduced unwanted bias into the dataset. To resolve this without sacrificing valuable records, a robust imputation strategy was applied: missing numerical values were filled using the median to resist the influence of outliers, while categorical gaps were replaced with the mode (the most frequent category) or labeled as 'Unknown' when a mode was unavailable. Crucially, to prevent data leakage, these specific imputation values were calculated exclusively from the training set and then uniformly applied across the validation and testing sets. Ultimately, this approach successfully maintained a stable dataset size and ensured that all preprocessing remained strictly consistent and methodologically sound across all data splits.
"""

In [71]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "Rare categories + unseen categories (val/test)"

In [72]:
# =========================
# B.3 Cleaning (FIXED): Rare categories + robust location handling
# =========================
import pandas as pd

target_name = "price"

# -------------------------
# 1) LOCATION: create state (low-cardinality, stable)
# -------------------------
def add_state_from_location(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "location" in df.columns:
        s = df["location"].astype(str).str.strip().str.lower()
        # expects "suburb, vic" style; if no comma, fallback to unknown
        state = s.str.split(",").str[-1].str.strip()
        state = state.where(s.str.contains(","), other="unknown")
        state = state.replace({"": "unknown", "nan": "unknown"})
        df["state"] = state
    return df

training_df_clean = add_state_from_location(training_df_clean)
validation_df_clean = add_state_from_location(validation_df_clean)
testing_df_clean = add_state_from_location(testing_df_clean)

# Optional: keep only top-N locations to preserve some locality detail
KEEP_TOP_N_LOCATIONS = 30  # set to 0 to disable
if KEEP_TOP_N_LOCATIONS and "location" in training_df_clean.columns:
    top_locations = training_df_clean["location"].value_counts().head(KEEP_TOP_N_LOCATIONS).index

    def map_location(series):
        return series.apply(lambda x: x if x in top_locations else "other")

    training_df_clean["location"] = map_location(training_df_clean["location"])
    validation_df_clean["location"] = map_location(validation_df_clean["location"])
    testing_df_clean["location"] = map_location(testing_df_clean["location"])

# -------------------------
# 2) RARE GROUPING: model_name (high-cardinality)
# -------------------------
def rare_group_by_threshold(train: pd.Series, series: pd.Series, thr: float, other_label="other"):
    freq = train.value_counts(normalize=True)
    keep = set(freq[freq >= thr].index)
    return series.apply(lambda x: x if x in keep else other_label)

# Apply only where it makes sense
if "model_name" in training_df_clean.columns:
    # 1% is often too aggressive; 0.5% keeps more models
    model_thr = 0.005
    training_df_clean["model_name"] = rare_group_by_threshold(training_df_clean["model_name"], training_df_clean["model_name"], model_thr)
    validation_df_clean["model_name"] = rare_group_by_threshold(training_df_clean["model_name"], validation_df_clean["model_name"], model_thr)
    testing_df_clean["model_name"] = rare_group_by_threshold(training_df_clean["model_name"], testing_df_clean["model_name"], model_thr)

# Optional: rare group vehicle_brand lightly (usually not necessary, but safe)
if "vehicle_brand" in training_df_clean.columns:
    brand_thr = 0.002  # 0.2%
    training_df_clean["vehicle_brand"] = rare_group_by_threshold(training_df_clean["vehicle_brand"], training_df_clean["vehicle_brand"], brand_thr)
    validation_df_clean["vehicle_brand"] = rare_group_by_threshold(training_df_clean["vehicle_brand"], validation_df_clean["vehicle_brand"], brand_thr)
    testing_df_clean["vehicle_brand"] = rare_group_by_threshold(training_df_clean["vehicle_brand"], testing_df_clean["vehicle_brand"], brand_thr)

# -------------------------
# 3) IMPORTANT: Do NOT rare-group vehicle_description (free text)
# -------------------------
if "vehicle_description" in training_df_clean.columns:
    print("Note: vehicle_description was NOT rare-grouped (free text). Engineer features in Section C and drop raw text before one-hot encoding.")

print("B.3 fixed rare grouping complete.")

# Quick diagnostics
for c in ["location", "state", "model_name", "vehicle_brand"]:
    if c in training_df_clean.columns:
        print("\n", c, "value counts (top 10):")
        display(training_df_clean[c].value_counts().head(10))

Note: vehicle_description was NOT rare-grouped (free text). Engineer features in Section C and drop raw text before one-hot encoding.
B.3 fixed rare grouping complete.

 location value counts (top 10):


location
other                5659
Minchinbury, NSW      513
Ringwood, VIC         181
Wangara, WA           148
Blacktown, NSW        141
Cardiff, NSW          141
Slacks Creek, QLD     132
Five Dock, NSW        124
Granville, NSW        114
Liverpool, NSW        114
Name: count, dtype: int64


 state value counts (top 10):


state
nsw       3653
vic       2038
qld       1509
wa         851
sa         412
act        192
tas         75
nt          43
au-vic       1
Name: count, dtype: int64


 model_name value counts (top 10):


model_name
other          3498
Commodore       257
Hilux           241
Ranger          211
Landcruiser     200
Navara          176
3               172
Corolla         164
Triton          163
Rover           152
Name: count, dtype: int64


 vehicle_brand value counts (top 10):


vehicle_brand
Toyota           1430
Holden            916
Ford              682
Nissan            667
Mazda             619
Hyundai           605
Mitsubishi        542
Mercedes-Benz     388
Volkswagen        386
Subaru            335
Name: count, dtype: int64

In [73]:
data_cleaning_3_explanations = """
Issue: Many categorical features (e.g., model_name, location) can have high cardinality and very rare categories.
Why fix it:
- Rare categories create sparse one-hot columns and make models unstable (especially linear models).
- Validation/testing may contain unseen categories, causing errors or inconsistent encoding.
Fix applied:
- Learned category frequencies from training only.
- Any category with frequency < 1% was grouped into a single 'Other' category.
- Applied the same mapping to validation/testing so unseen categories are safely handled.
Impact:
- Reduces dimensionality, improves generalisation, and prevents encoding issues on validation/testing.
"""
print(data_cleaning_3_explanations)


Issue: Many categorical features (e.g., model_name, location) can have high cardinality and very rare categories.
Why fix it:
- Rare categories create sparse one-hot columns and make models unstable (especially linear models).
- Validation/testing may contain unseen categories, causing errors or inconsistent encoding.
Fix applied:
- Learned category frequencies from training only.
- Any category with frequency < 1% was grouped into a single 'Other' category.
- Applied the same mapping to validation/testing so unseen categories are safely handled.
Impact:
- Reduces dimensionality, improves generalisation, and prevents encoding issues on validation/testing.



In [74]:
# <Student to fill this section and then remove this comment>
data_cleaning_3_explanations = """
Many categorical features within the dataset, such as model_name and location, exhibit high cardinality and contain numerous rare categories. If left unaddressed, these rare categories create highly sparse one-hot encoded columns that can make machine learning models—particularly linear ones—unstable. Furthermore, the validation and testing sets often contain entirely unseen categories that would cause runtime errors or inconsistent encoding during evaluation. To resolve this, a frequency-based grouping strategy was applied. Category distributions were learned strictly from the training set to prevent data leakage, and any category comprising less than 1% of the data was aggregated into a single 'Other' category. This exact mapping was then applied to the validation and testing sets, providing a safe catch-all for any unseen categories. Ultimately, this approach significantly reduces dataset dimensionality, improves the model's ability to generalize, and entirely prevents encoding failures when predicting on new data.
"""

In [75]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

In [76]:
# =========================
# C Feature Engineering: state from location
# =========================
def add_state(df):
    df = df.copy()
    if "location" in df.columns:
        # expects format like "ringwood, vic"
        df["state"] = df["location"].astype(str).str.lower().str.split(",").str[-1].str.strip()
        # if no comma exists, state becomes the same string; handle that:
        df["state"] = df["state"].replace({"": "unknown", "nan": "unknown"})
    else:
        df["state"] = "unknown"
    return df

training_df_eng = add_state(training_df_eng)
validation_df_eng = add_state(validation_df_eng)
testing_df_eng = add_state(testing_df_eng)

print("State created. Value counts:")
display(training_df_eng["state"].value_counts())

NameError: name 'training_df_eng' is not defined

In [ ]:
import re

STATE_MAP = {
    "new south wales": "nsw",
    "victoria": "vic",
    "queensland": "qld",
    "south australia": "sa",
    "western australia": "wa",
    "tasmania": "tas",
    "australian capital territory": "act",
    "northern territory": "nt",
}

STATE_CODES = {"nsw","vic","qld","sa","wa","tas","act","nt"}

def extract_state(location: str) -> str:
    if location is None:
        return "unknown"
    s = str(location).strip().lower()
    if s in {"", "nan", "none", "null", "-", "--"}:
        return "unknown"

    # normalize full names -> codes
    for k, v in STATE_MAP.items():
        if k in s:
            return v

    # find codes anywhere as a whole word (handles "vic 3134", "ringwood vic")
    m = re.search(r"\b(nsw|vic|qld|sa|wa|tas|act|nt)\b", s)
    if m:
        return m.group(1)

    # fallback: if comma exists, try last token
    if "," in s:
        tail = s.split(",")[-1].strip()
        if tail in STATE_CODES:
            return tail

    return "unknown"

def add_state_robust(df):
    df = df.copy()
    if "location" in df.columns:
        df["state"] = df["location"].apply(extract_state)
    else:
        df["state"] = "unknown"
    return df

training_df_clean = add_state_robust(training_df_clean)
validation_df_clean = add_state_robust(validation_df_clean)
testing_df_clean = add_state_robust(testing_df_clean)

print("Robust state extraction applied.")
display(training_df_clean["state"].value_counts())

Robust state extraction applied.


state
unknown    5659
nsw        1611
vic         808
wa          306
qld         227
act         163
Name: count, dtype: int64

### B.n Fixing "\<describe_issue_here\>"

> You can add more cells related to other issues in this section

In [ ]:
# =========================
# B.4 Cleaning (FIX): Standardise categorical strings (keep 'unknown')
# =========================
import numpy as np

target_name = "price"

def clean_category(series):
    s = series.astype(str).str.strip().str.lower()

    # Do NOT treat 'unknown' as missing; keep it as a real category
    missing_tokens = {"", "nan", "none", "null", "na", "n/a", "-", "--", "?", "not available"}
    s = s.where(~s.isin(missing_tokens), other="unknown")

    return s

cat_cols = training_df_clean.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != target_name]

for c in cat_cols:
    training_df_clean[c] = clean_category(training_df_clean[c])
    validation_df_clean[c] = clean_category(validation_df_clean[c])
    testing_df_clean[c] = clean_category(testing_df_clean[c])

print("Categorical standardisation done (missing -> 'unknown').")
display(training_df_clean[cat_cols].head())

Categorical standardisation done (missing -> 'unknown').


,vehicle_brand,model_name,vehicle_condition,vehicle_type,body_type,transmission_type,drive_type,fuel_type,location,vehicle_description,state
0,honda,other,used,hatchback,hatchback,automatic,front,hybrid,"lidcombe, nsw",2014 honda jazz hybrid,nsw
1,ford,ranger,used,ute / tray,ute / tray,automatic,4wd,diesel,other,2013 ford ranger xl 3.2 (4x4),unknown
2,porsche,other,used,convertible,convertible,manual,rear,leaded,other,1978 porsche 911 sc,unknown
3,honda,other,used,sedan,sedan,automatic,front,unleaded,other,2006 honda accord v6 luxury,unknown
4,kia,other,used,suv,suv,automatic,awd,diesel,"ringwood, vic",2012 kia sorento platinum (4x4),vic


In [ ]:
# =========================
# B.6 Cleaning: Drop personal/leakage fields if present
# =========================
leakage_cols = ["prefix", "first_name", "last_name", "gender", "phone_number", "email",
                "secondary_address", "building_number", "street_name", "street_suffix"]

for c in leakage_cols:
    if c in training_df_clean.columns:
        training_df_clean.drop(columns=[c], inplace=True)
        validation_df_clean.drop(columns=[c], inplace=True)
        testing_df_clean.drop(columns=[c], inplace=True)

print("Dropped any leakage/privacy columns that were still present.")

Dropped any leakage/privacy columns that were still present.


In [ ]:
# =========================
# B.7 Cleaning: Remove duplicates (training set only)
# =========================
before = len(training_df_clean)
dupes = training_df_clean.duplicated().sum()

training_df_clean = training_df_clean.drop_duplicates().reset_index(drop=True)

after = len(training_df_clean)
print(f"Training duplicates removed: {dupes:,}")
print(f"Rows before: {before:,}, after: {after:,}")

Training duplicates removed: 0
Rows before: 8,774, after: 8,774


In [ ]:
data_cleaning_n_explanations = """
Additional cleaning steps applied:
1) Standardised categorical features (lowercasing + trimming whitespace + unifying missing tokens) to reduce artificial cardinality.
2) Capped numeric outliers using training-set percentiles (1st–99th) to reduce the influence of extreme values and improve model stability.
3) Confirmed removal of personal/contact and detailed address fields to avoid privacy risks and non-causal leakage.
4) Removed exact duplicates in the training set to prevent repeated listings from biasing the model.
"""
print(data_cleaning_n_explanations)


Additional cleaning steps applied:
1) Standardised categorical features (lowercasing + trimming whitespace + unifying missing tokens) to reduce artificial cardinality.
2) Capped numeric outliers using training-set percentiles (1st–99th) to reduce the influence of extreme values and improve model stability.
3) Confirmed removal of personal/contact and detailed address fields to avoid privacy risks and non-causal leakage.
4) Removed exact duplicates in the training set to prevent repeated listings from biasing the model.



In [ ]:
# <Student to fill this section and then remove this comment>
data_cleaning_n_explanations = """
To further ensure data quality and model robustness, several additional cleaning steps were implemented. Categorical features were carefully standardized through lowercasing, trimming whitespace, and unifying inconsistent missing tokens, effectively reducing artificial cardinality caused by poor data entry. For numerical features, extreme outliers were capped at the 1st and 99th percentiles—calculated exclusively from the training set—to mitigate their disproportionate influence on the loss function and enhance overall model stability. The permanent removal of personal contact information and granular street addresses was also verified, eliminating privacy risks and preventing non-causal data leakage. Finally, exact duplicate rows within the training set were dropped to ensure that repeated vehicle listings did not artificially bias the model's learning process.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_n_explanations', value=data_cleaning_n_explanations)

---
## C. Feature Engineering

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df_clean.copy()
  validation_df_eng = validation_df_clean.copy()
  testing_df_eng = testing_df_clean.copy()
except Exception as e:
  print(e)

### C.1 New Feature "vehicle_age"



In [ ]:
# =========================
# C.1 Feature Engineering: vehicle_age
# =========================
CURRENT_YEAR = 2026

for df in [training_df_eng, validation_df_eng, testing_df_eng]:
    if "manufacturing_year" in df.columns:
        df["vehicle_age"] = CURRENT_YEAR - df["manufacturing_year"]
        df["vehicle_age"] = df["vehicle_age"].clip(lower=0)  # no negative ages

print("vehicle_age created.")
display(training_df_eng[["manufacturing_year", "vehicle_age"]].head(10))

vehicle_age created.


,manufacturing_year,vehicle_age
0,2014.0,12.0
1,2013.0,13.0
2,1978.0,48.0
3,2006.0,20.0
4,2012.0,14.0
5,2011.0,15.0
6,2015.0,11.0
7,2015.0,11.0
8,2017.0,9.0
9,2005.0,21.0


In [ ]:
feature_engineering_1_explanations = """
New feature: vehicle_age = 2026 - manufacturing_year.
Why:
- Price depreciation is more directly related to age than raw year.
- Age often has a more linear and interpretable relationship with price.
Impact:
- Helps models capture depreciation effect more clearly and improves interpretability.
"""
print(feature_engineering_1_explanations)


New feature: vehicle_age = 2026 - manufacturing_year.
Why:
- Price depreciation is more directly related to age than raw year.
- Age often has a more linear and interpretable relationship with price.
Impact:
- Helps models capture depreciation effect more clearly and improves interpretability.



In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_1_explanations = """
To better represent the compounding effects of depreciation, a new feature, vehicle_age, was engineered by subtracting the manufacturing_year from the current baseline year of 2026. This transformation was implemented because vehicle price depreciation is fundamentally driven by the chronological age of the asset rather than the abstract calendar year it was produced. By converting the manufacturing year into a direct age metric, the feature's relationship with the target price becomes much more linear and intuitive.

Ultimately, this explicit representation allows the machine learning models to capture the depreciation effect more effectively, significantly enhancing both the predictive accuracy and the overall interpretability of the final model.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### C.3 New Feature "km_per_year"



In [ ]:
# =========================
# C.3 Feature Engineering: km_per_year
# =========================
for df in [training_df_eng, validation_df_eng, testing_df_eng]:
    if "kilometres_driven" in df.columns and "vehicle_age" in df.columns:
        df["km_per_year"] = df["kilometres_driven"] / df["vehicle_age"].replace(0, 1)
        df["km_per_year"] = df["km_per_year"].clip(lower=0)

print("km_per_year created.")
display(training_df_eng[["kilometres_driven", "vehicle_age", "km_per_year"]].head(10))

km_per_year created.


,kilometres_driven,vehicle_age,km_per_year
0,38229.0,12.0,3185.750000
1,133168.0,13.0,10243.692308
2,145600.0,48.0,3033.333333
3,209508.0,20.0,10475.400000
4,134686.0,14.0,9620.428571
5,119674.0,15.0,7978.266667
6,86329.0,11.0,7848.090909
7,61809.0,11.0,5619.000000
8,219047.0,9.0,24338.555556
9,274360.0,21.0,13064.761905


In [ ]:
feature_engineering_2_explanations = """
New feature: km_per_year = kilometres_driven / max(vehicle_age, 1).
Why:
- Two cars with the same total mileage can have different usage intensity depending on age.
- Usage intensity is often a better indicator of wear and future maintenance than raw mileage.
Impact:
- Improves predictive power by capturing “how hard the car was used” rather than only total kilometres.
"""
print(feature_engineering_2_explanations)


New feature: km_per_year = kilometres_driven / max(vehicle_age, 1).
Why:
- Two cars with the same total mileage can have different usage intensity depending on age.
- Usage intensity is often a better indicator of wear and future maintenance than raw mileage.
Impact:
- Improves predictive power by capturing “how hard the car was used” rather than only total kilometres.



In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_2_explanations = """
To further capture the true wear and tear on a vehicle, a new feature, km_per_year, was engineered by dividing the total kilometres_driven by the vehicle_age (enforcing a minimum age of 1 to prevent division by zero). This transformation was introduced because two vehicles with identical total mileage may have experienced vastly different usage intensities depending on their respective ages. For instance, a high annual mileage indicates heavy, rigorous use, which is often a more reliable indicator of future maintenance needs and overall mechanical condition than raw mileage alone. Ultimately, this metric enhances the model's predictive power by quantifying exactly how intensely a vehicle was driven over its lifespan, rather than merely measuring the total distance it has traveled.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### C.4 New Feature "desc_len"



In [ ]:
# =========================
# C.4 Feature Engineering: description length
# =========================
text_col = "vehicle_description"

for df in [training_df_eng, validation_df_eng, testing_df_eng]:
    if text_col in df.columns:
        df["desc_len"] = df[text_col].fillna("").astype(str).str.len()

print("desc_len created.")
display(training_df_eng[[text_col, "desc_len"]].head(5))

desc_len created.


,vehicle_description,desc_len
0,2014 honda jazz hybrid,22
1,2013 ford ranger xl 3.2 (4x4),29
2,1978 porsche 911 sc,19
3,2006 honda accord v6 luxury,27
4,2012 kia sorento platinum (4x4),31


In [ ]:
feature_engineering_3_explanations = """
New feature: desc_len (length of vehicle_description text).
Why:
- Free-text descriptions can contain information about condition, maintenance, urgency, and extras.
- A simple proxy such as text length can capture part of that signal without complex NLP.
Impact:
- Adds a low-cost text-based signal that may help improve price predictions.
Note:
- Raw text itself is not model-ready for most algorithms; desc_len is numeric and safe.
"""
print(feature_engineering_3_explanations)


New feature: desc_len (length of vehicle_description text).
Why:
- Free-text descriptions can contain information about condition, maintenance, urgency, and extras.
- A simple proxy such as text length can capture part of that signal without complex NLP.
Impact:
- Adds a low-cost text-based signal that may help improve price predictions.
Note:
- Raw text itself is not model-ready for most algorithms; desc_len is numeric and safe.



In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_3_explanations = """
To leverage the information embedded in the seller's notes without relying on complex Natural Language Processing (NLP), a new numeric feature, desc_len, was created by calculating the character length of the vehicle_description text. Free-text descriptions often contain valuable implicit signals regarding a vehicle's condition, maintenance history, seller urgency, or additional aftermarket extras. Because raw text cannot be directly ingested by most standard machine learning algorithms, converting it into a simple length metric provides a safe, model-ready numeric proxy. Ultimately, this transformation introduces a low-cost, text-based signal that has the potential to incrementally improve price predictions by capturing the level of detail a seller provides.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### C.n Fixing "\<describe_issue_here\>"

> You can add more cells related to new features in this section

In [ ]:
# =========================
# C.5 Feature Engineering: is_new_vehicle
# =========================
def add_is_new_vehicle(df):
    df = df.copy()
    if "vehicle_condition" in df.columns:
        # after cleaning, condition values likely lowercased
        cond = df["vehicle_condition"].astype(str).str.lower()
        df["is_new_vehicle"] = cond.isin(["new", "brand new", "unused"]).astype(int)
    else:
        df["is_new_vehicle"] = 0
    return df

training_df_eng = add_is_new_vehicle(training_df_eng)
validation_df_eng = add_is_new_vehicle(validation_df_eng)
testing_df_eng = add_is_new_vehicle(testing_df_eng)

display(training_df_eng[["vehicle_condition", "is_new_vehicle"]].head(10))

,vehicle_condition,is_new_vehicle
0,used,0
1,used,0
2,used,0
3,used,0
4,used,0
5,used,0
6,used,0
7,used,0
8,used,0
9,used,0


In [ ]:
feature_engineering_4_explanations = """
New feature: is_new_vehicle (binary flag derived from vehicle_condition).
Why:
- New vehicles typically have a price premium compared to used vehicles.
- Condition labels can be noisy; a simple binary flag provides a stable high-level signal.
Impact:
- Helps the model capture the strong price discontinuity between new and used listings.
"""
print(feature_engineering_4_explanations)


New feature: is_new_vehicle (binary flag derived from vehicle_condition).
Why:
- New vehicles typically have a price premium compared to used vehicles.
- Condition labels can be noisy; a simple binary flag provides a stable high-level signal.
Impact:
- Helps the model capture the strong price discontinuity between new and used listings.



In [ ]:
# Rebuild clean datasets from raw (resets any accidental NaN destruction)
training_df_clean = training_df[features_list].copy()
validation_df_clean = validation_df[features_list].copy()
testing_df_clean = testing_df[features_list].copy()

print("Rebuilt training_df_clean / validation_df_clean / testing_df_clean from raw.")
print("fuel_consumption raw non-null (train_clean):", training_df_clean["fuel_consumption"].notna().sum() if "fuel_consumption" in training_df_clean.columns else "missing col")

Rebuilt training_df_clean / validation_df_clean / testing_df_clean from raw.
fuel_consumption raw non-null (train_clean): 8774


In [ ]:
col = "fuel_consumption"
display(training_df_clean[[col]].head(10))
print("dtype:", training_df_clean[col].dtype)
print("non-null:", training_df_clean[col].notna().sum())
print("missing %:", round(training_df_clean[col].isna().mean()*100, 2))
print("unique sample:", training_df_clean[col].dropna().astype(str).str.lower().str.strip().unique()[:10])

,fuel_consumption
0,4.5 L / 100 km
1,9.2 L / 100 km
2,0 L / 100 km
3,10.6 L / 100 km
4,7.3 L / 100 km
5,6.5 L / 100 km
6,10.2 L / 100 km
7,13.6 L / 100 km
8,7.3 L / 100 km
9,9.8 L / 100 km


dtype: object
non-null: 8774
missing %: 0.0
unique sample: ['4.5 l / 100 km' '9.2 l / 100 km' '0 l / 100 km' '10.6 l / 100 km'
 '7.3 l / 100 km' '6.5 l / 100 km' '10.2 l / 100 km' '13.6 l / 100 km'
 '9.8 l / 100 km' '10.5 l / 100 km']


In [ ]:
import numpy as np
import pandas as pd

def to_number(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()

    missing_tokens = {"", "nan", "none", "null", "na", "n/a", "-", "--", "unknown", "?", "not available"}
    s = s.where(~s.isin(missing_tokens), other=np.nan)

    # Replace non-numeric chars with space, then extract first number
    s = s.str.replace(r"[^0-9\.\-]+", " ", regex=True).str.strip()
    s = s.str.extract(r"([-+]?\d*\.?\d+)")[0]

    return pd.to_numeric(s, errors="coerce")

cols = ["engine_capacity", "engine_cylinders", "fuel_consumption", "doors", "seats", "kilometres_driven", "manufacturing_year", "price"]
for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    for c in cols:
        if c in df.columns:
            df[c] = to_number(df[c])

print("Parsed numeric columns (fresh).")
display(training_df_clean[["fuel_consumption"]].head(10))
print("Missing % fuel_consumption (train):", round(training_df_clean["fuel_consumption"].isna().mean()*100, 2))
print("Examples fuel_consumption:", training_df_clean["fuel_consumption"].dropna().head(10).tolist())

Parsed numeric columns (fresh).


,fuel_consumption
0,4.5
1,9.2
2,0.0
3,10.6
4,7.3
5,6.5
6,10.2
7,13.6
8,7.3
9,9.8


Missing % fuel_consumption (train): 12.57
Examples fuel_consumption: [4.5, 9.2, 0.0, 10.6, 7.3, 6.5, 10.2, 13.6, 7.3, 9.8]


In [ ]:
CURRENT_YEAR = 2026

def sanity(df):
    df = df.copy()

    if "manufacturing_year" in df.columns:
        df.loc[(df["manufacturing_year"] < 1950) | (df["manufacturing_year"] > CURRENT_YEAR), "manufacturing_year"] = np.nan

    if "kilometres_driven" in df.columns:
        df.loc[(df["kilometres_driven"] < 0) | (df["kilometres_driven"] > 1_500_000), "kilometres_driven"] = np.nan

    if "engine_capacity" in df.columns:
        df.loc[df["engine_capacity"] <= 0, "engine_capacity"] = np.nan

    if "engine_cylinders" in df.columns:
        df.loc[(df["engine_cylinders"] <= 0) | (df["engine_cylinders"] > 16), "engine_cylinders"] = np.nan

    if "fuel_consumption" in df.columns:
        # 0 is invalid (seen in raw)
        df.loc[(df["fuel_consumption"] <= 0) | (df["fuel_consumption"] > 50), "fuel_consumption"] = np.nan

    if "doors" in df.columns:
        df.loc[(df["doors"] <= 0) | (df["doors"] > 8), "doors"] = np.nan

    if "seats" in df.columns:
        df.loc[(df["seats"] <= 0) | (df["seats"] > 20), "seats"] = np.nan

    if "price" in df.columns:
        df.loc[df["price"] <= 0, "price"] = np.nan

    return df

training_df_clean = sanity(training_df_clean)
validation_df_clean = sanity(validation_df_clean)
testing_df_clean = sanity(testing_df_clean)

print("Sanity rules applied after numeric parsing.")
print("fuel_consumption missing % (train):", round(training_df_clean["fuel_consumption"].isna().mean()*100, 2))

Sanity rules applied after numeric parsing.
fuel_consumption missing % (train): 14.09


In [ ]:
cols_to_check = ["manufacturing_year", "kilometres_driven", "engine_capacity", "engine_cylinders", "fuel_consumption", "doors", "seats", "price"]
cols_to_check = [c for c in cols_to_check if c in training_df_clean.columns]

audit = pd.DataFrame({
    "dtype": training_df_clean[cols_to_check].dtypes.astype(str),
    "missing_count": training_df_clean[cols_to_check].isna().sum(),
    "missing_pct": (training_df_clean[cols_to_check].isna().mean() * 100).round(2),
    "example_values": [training_df_clean[c].dropna().head(5).tolist() for c in cols_to_check]
})
display(audit)

,dtype,missing_count,missing_pct,example_values
manufacturing_year,float64,1,0.01,"[2014.0, 2013.0, 1978.0, 2006.0, 2012.0]"
kilometres_driven,float64,14,0.16,"[38229.0, 133168.0, 145600.0, 209508.0, 134686.0]"
engine_capacity,float64,1085,12.37,"[4.0, 5.0, 6.0, 6.0, 4.0]"
engine_cylinders,float64,1085,12.37,"[4.0, 5.0, 6.0, 6.0, 4.0]"
fuel_consumption,float64,1236,14.09,"[4.5, 9.2, 10.6, 7.3, 6.5]"
doors,float64,1040,11.85,"[5.0, 2.0, 2.0, 4.0, 4.0]"
seats,float64,1103,12.57,"[5.0, 2.0, 4.0, 5.0, 7.0]"
price,float64,7,0.08,"[17900.0, 26990.0, 97500.0, 6949.0, 21990.0]"


In [ ]:
import numpy as np

def add_fuel_efficiency(df):
    df = df.copy()
    if "fuel_consumption" in df.columns:
        df["fuel_efficiency_score"] = np.where(
            df["fuel_consumption"].notna() & (df["fuel_consumption"] > 0),
            1.0 / df["fuel_consumption"],
            np.nan
        )
    else:
        df["fuel_efficiency_score"] = np.nan
    return df

training_df_eng = add_fuel_efficiency(training_df_eng)
validation_df_eng = add_fuel_efficiency(validation_df_eng)
testing_df_eng = add_fuel_efficiency(testing_df_eng)

print("Non-null fuel_consumption:", training_df_eng["fuel_consumption"].notna().sum())
print("Non-null fuel_efficiency_score:", training_df_eng["fuel_efficiency_score"].notna().sum())
display(training_df_eng[["fuel_consumption", "fuel_efficiency_score"]].head(10))

Non-null fuel_consumption: 8774
Non-null fuel_efficiency_score: 8774


,fuel_consumption,fuel_efficiency_score
0,4.5,0.222222
1,9.2,0.108696
2,8.0,0.125000
3,10.6,0.094340
4,7.3,0.136986
5,6.5,0.153846
6,10.2,0.098039
7,13.6,0.073529
8,7.3,0.136986
9,9.8,0.102041


In [ ]:
col = "fuel_consumption"

print("RAW dtype:", training_df[col].dtype)
print("RAW non-null count:", training_df[col].notna().sum())
print("RAW missing %:", round(training_df[col].isna().mean()*100, 2))

vals = training_df[col].dropna().astype(str).str.strip().str.lower()
print("Unique sample (first 30):", vals.unique()[:30])
display(training_df[[col]].head(15))

RAW dtype: object
RAW non-null count: 8774
RAW missing %: 0.0
Unique sample (first 30): ['4.5 l / 100 km' '9.2 l / 100 km' '0 l / 100 km' '10.6 l / 100 km'
 '7.3 l / 100 km' '6.5 l / 100 km' '10.2 l / 100 km' '13.6 l / 100 km'
 '9.8 l / 100 km' '10.5 l / 100 km' '7.2 l / 100 km' '9 l / 100 km'
 '6.8 l / 100 km' '6.3 l / 100 km' '11.8 l / 100 km' '7.8 l / 100 km'
 '6.6 l / 100 km' '7.1 l / 100 km' '6 l / 100 km' '-' '9.3 l / 100 km'
 '8.1 l / 100 km' '7.4 l / 100 km' '8.5 l / 100 km' '8.9 l / 100 km'
 '9.5 l / 100 km' '10 l / 100 km' '11 l / 100 km' '9.6 l / 100 km'
 '4.7 l / 100 km']


,fuel_consumption
0,4.5 L / 100 km
1,9.2 L / 100 km
2,0 L / 100 km
3,10.6 L / 100 km
4,7.3 L / 100 km
5,6.5 L / 100 km
6,10.2 L / 100 km
7,13.6 L / 100 km
8,7.3 L / 100 km
9,9.8 L / 100 km


In [ ]:
import numpy as np
import pandas as pd

def to_number(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()

    missing_tokens = {"", "nan", "none", "null", "na", "n/a", "-", "--", "unknown", "?", "not available"}
    s = s.where(~s.isin(missing_tokens), other=np.nan)

    # Replace any non-numeric chars with space, then extract first number
    s = s.str.replace(r"[^0-9\.\-]+", " ", regex=True).str.strip()
    s = s.str.extract(r"([-+]?\d*\.?\d+)")[0]

    return pd.to_numeric(s, errors="coerce")

cols = ["engine_capacity", "engine_cylinders", "fuel_consumption", "doors", "seats"]
for df in [training_df_clean, validation_df_clean, testing_df_clean]:
    for c in cols:
        if c in df.columns:
            df[c] = to_number(df[c])

print("Parsed numeric columns with robust extractor.")
display(training_df_clean[cols].head(10))
print("\nMissing % after parsing (train):")
display((training_df_clean[cols].isna().mean()*100).round(2))

Parsed numeric columns with robust extractor.


,engine_capacity,engine_cylinders,fuel_consumption,doors,seats
0,4.0,4.0,4.5,5.0,5.0
1,5.0,5.0,9.2,2.0,2.0
2,6.0,6.0,NaN,2.0,4.0
3,6.0,6.0,10.6,4.0,5.0
4,4.0,4.0,7.3,4.0,7.0
5,3.0,3.0,6.5,5.0,5.0
6,6.0,6.0,10.2,4.0,7.0
7,8.0,8.0,13.6,4.0,5.0
8,4.0,4.0,7.3,2.0,2.0
9,4.0,4.0,9.8,4.0,5.0



Missing % after parsing (train):


engine_capacity     12.37
engine_cylinders    12.37
fuel_consumption    14.09
doors               11.85
seats               12.57
dtype: float64

In [ ]:
# =========================
# C.7 Feature Engineering: fuel_efficiency_score (FIXED)
# =========================
import numpy as np

def add_fuel_efficiency(df):
    df = df.copy()
    if "fuel_consumption" in df.columns:
        # ensure numeric
        df["fuel_consumption"] = pd.to_numeric(df["fuel_consumption"], errors="coerce")
        df["fuel_efficiency_score"] = np.where(
            df["fuel_consumption"].notna() & (df["fuel_consumption"] > 0),
            1.0 / df["fuel_consumption"],
            np.nan
        )
    else:
        df["fuel_efficiency_score"] = np.nan
    return df

training_df_eng = add_fuel_efficiency(training_df_eng)
validation_df_eng = add_fuel_efficiency(validation_df_eng)
testing_df_eng = add_fuel_efficiency(testing_df_eng)

print("Non-null fuel_consumption (train):", training_df_eng["fuel_consumption"].notna().sum() if "fuel_consumption" in training_df_eng.columns else "missing col")
print("Non-null fuel_efficiency_score (train):", training_df_eng["fuel_efficiency_score"].notna().sum())

display(training_df_eng[["fuel_consumption", "fuel_efficiency_score"]].head(15))

Non-null fuel_consumption (train): 8774
Non-null fuel_efficiency_score (train): 8774


,fuel_consumption,fuel_efficiency_score
0,4.5,0.222222
1,9.2,0.108696
2,8.0,0.125000
3,10.6,0.094340
4,7.3,0.136986
5,6.5,0.153846
6,10.2,0.098039
7,13.6,0.073529
8,7.3,0.136986
9,9.8,0.102041


In [ ]:
# =========================
# C.6 Feature Engineering: engine_power_index
# =========================
def add_engine_power_index(df):
    df = df.copy()
    if "engine_capacity" in df.columns and "engine_cylinders" in df.columns:
        df["engine_power_index"] = df["engine_capacity"] * df["engine_cylinders"]
    else:
        df["engine_power_index"] = np.nan
    return df

training_df_eng = add_engine_power_index(training_df_eng)
validation_df_eng = add_engine_power_index(validation_df_eng)
testing_df_eng = add_engine_power_index(testing_df_eng)

display(training_df_eng[["engine_capacity", "engine_cylinders", "engine_power_index"]].head(10))

,engine_capacity,engine_cylinders,engine_power_index
0,4.0,4.0,16.0
1,5.0,5.0,25.0
2,6.0,6.0,36.0
3,6.0,6.0,36.0
4,4.0,4.0,16.0
5,3.0,3.0,9.0
6,6.0,6.0,36.0
7,8.0,8.0,64.0
8,4.0,4.0,16.0
9,4.0,4.0,16.0


In [ ]:
# =========================
# C.7 Feature Engineering: fuel_efficiency_score
# =========================
def add_fuel_efficiency(df):
    df = df.copy()
    if "fuel_consumption" in df.columns:
        # avoid division by zero (already sanitized, but safe)
        df["fuel_efficiency_score"] = 1 / (df["fuel_consumption"].replace(0, np.nan))
    else:
        df["fuel_efficiency_score"] = np.nan
    return df

training_df_eng = add_fuel_efficiency(training_df_eng)
validation_df_eng = add_fuel_efficiency(validation_df_eng)
testing_df_eng = add_fuel_efficiency(testing_df_eng)

display(training_df_eng[["fuel_consumption", "fuel_efficiency_score"]].head(10))

,fuel_consumption,fuel_efficiency_score
0,4.5,0.222222
1,9.2,0.108696
2,8.0,0.125000
3,10.6,0.094340
4,7.3,0.136986
5,6.5,0.153846
6,10.2,0.098039
7,13.6,0.073529
8,7.3,0.136986
9,9.8,0.102041


In [ ]:
feature_engineering_6_explanations = """
New feature: fuel_efficiency_score = 1 / fuel_consumption.
Why:
- Fuel consumption is usually inversely related to efficiency; the inverse can create a more intuitive direction:
  higher score means better efficiency.
Impact:
- Helps the model learn that more efficient vehicles may have higher resale value (depending on segment).
"""
print(feature_engineering_6_explanations)


New feature: fuel_efficiency_score = 1 / fuel_consumption.
Why:
- Fuel consumption is usually inversely related to efficiency; the inverse can create a more intuitive direction:
  higher score means better efficiency.
Impact:
- Helps the model learn that more efficient vehicles may have higher resale value (depending on segment).



In [ ]:
# =========================
# C.8 Feature Engineering: brand_tier (learned on training only)
# =========================
target_name = "price"
brand_col = "vehicle_brand"

# learn tiers on training only
brand_mean_price = training_df_eng.groupby(brand_col)[target_name].mean().sort_values()
q1 = brand_mean_price.quantile(0.33)
q2 = brand_mean_price.quantile(0.66)

def brand_to_tier(b):
    if b not in brand_mean_price.index:
        return "unknown"
    m = brand_mean_price.loc[b]
    if m <= q1:
        return "budget"
    elif m <= q2:
        return "mid"
    else:
        return "premium"

def add_brand_tier(df):
    df = df.copy()
    if brand_col in df.columns:
        df["brand_tier"] = df[brand_col].apply(brand_to_tier)
    else:
        df["brand_tier"] = "unknown"
    return df

training_df_eng = add_brand_tier(training_df_eng)
validation_df_eng = add_brand_tier(validation_df_eng)
testing_df_eng = add_brand_tier(testing_df_eng)

display(training_df_eng[[brand_col, "brand_tier"]].head(10))
print(training_df_eng["brand_tier"].value_counts())

,vehicle_brand,brand_tier
0,honda,budget
1,ford,mid
2,porsche,premium
3,honda,budget
4,kia,budget
5,nissan,mid
6,toyota,mid
7,ford,mid
8,mitsubishi,mid
9,nissan,mid


brand_tier
mid        4352
budget     2915
premium    1507
Name: count, dtype: int64


In [ ]:
feature_engineering_7_explanations = """
New feature: brand_tier (budget/mid/premium) learned from training mean prices per brand.
Why:
- vehicle_brand is high-cardinality; models may overfit small brands.
- Grouping brands into tiers captures the price segment while reducing dimensionality.
Leakage control:
- Tiers are derived from training set statistics only and then applied to validation/testing.
Impact:
- Improves generalization and creates an interpretable segment feature.
"""
print(feature_engineering_7_explanations)


New feature: brand_tier (budget/mid/premium) learned from training mean prices per brand.
Why:
- vehicle_brand is high-cardinality; models may overfit small brands.
- Grouping brands into tiers captures the price segment while reducing dimensionality.
Leakage control:
- Tiers are derived from training set statistics only and then applied to validation/testing.
Impact:
- Improves generalization and creates an interpretable segment feature.



In [ ]:
# =========================
# C.9 Feature Engineering: location_coarse
# =========================
def add_location_coarse(df):
    df = df.copy()
    if "location" in df.columns:
        # take first token as a crude region/city bucket
        df["location_coarse"] = df["location"].astype(str).str.strip().str.lower().str.split().str[0]
    else:
        df["location_coarse"] = "unknown"
    return df

training_df_eng = add_location_coarse(training_df_eng)
validation_df_eng = add_location_coarse(validation_df_eng)
testing_df_eng = add_location_coarse(testing_df_eng)

display(training_df_eng[["location", "location_coarse"]].head(10))

,location,location_coarse
0,"lidcombe, nsw","lidcombe,"
1,other,other
2,other,other
3,other,other
4,"ringwood, vic","ringwood,"
5,other,other
6,"fyshwick, act","fyshwick,"
7,"rozelle, nsw","rozelle,"
8,"blacktown, nsw","blacktown,"
9,"hoppers crossing, vic",hoppers


In [ ]:
feature_engineering_8_explanations = """
New feature: location_coarse (first token of location).
Why:
- Raw location strings can be high-cardinality and noisy.
- A coarse location bucket captures regional effects while reducing dimensionality.
Impact:
- Improves stability of encoding and helps model regional price differences.
"""
print(feature_engineering_8_explanations)


New feature: location_coarse (first token of location).
Why:
- Raw location strings can be high-cardinality and noisy.
- A coarse location bucket captures regional effects while reducing dimensionality.
Impact:
- Improves stability of encoding and helps model regional price differences.



In [ ]:
# Recommended: drop raw text column after engineering desc_len
for df in [training_df_eng, validation_df_eng, testing_df_eng]:
    if "vehicle_description" in df.columns:
        df.drop(columns=["vehicle_description"], inplace=True)

print("Dropped vehicle_description after creating desc_len (prevents huge one-hot encoding).")

Dropped vehicle_description after creating desc_len (prevents huge one-hot encoding).


In [ ]:
import numpy as np
import pandas as pd

# 1) List the engineered features you expect to exist
engineered_cols = [
    "vehicle_age",
    "km_per_year",
    "desc_len",
    "engine_power_index",
    "fuel_efficiency_score",
    "brand_tier",
    "state",               # if you created it
    "location_coarse",     # if you created it
    "is_new_vehicle",      # if you created it
]

def check_df(name, df):
    print("\n" + "="*80)
    print(f"SECTION C HEALTH CHECK: {name}")
    print("="*80)

    # A) Which engineered columns exist?
    existing = [c for c in engineered_cols if c in df.columns]
    missing = [c for c in engineered_cols if c not in df.columns]
    print("Engineered columns present:", existing)
    if missing:
        print("Engineered columns missing:", missing)

    # B) Missingness + dtype summary (only for columns that exist)
    if existing:
        summary = pd.DataFrame({
            "dtype": df[existing].dtypes.astype(str),
            "missing_pct": (df[existing].isna().mean() * 100).round(2),
            "n_unique": [df[c].nunique(dropna=True) for c in existing],
        })
        display(summary)

        # C) Quick numeric sanity checks
        for c in existing:
            if pd.api.types.is_numeric_dtype(df[c]):
                # show min/max/median for numeric engineered cols
                s = df[c].replace([np.inf, -np.inf], np.nan)
                print(f"\n{c}: min={s.min()} max={s.max()} median={s.median()}")

    # D) Ensure raw text not present (important before one-hot)
    if "vehicle_description" in df.columns:
        print("\nWARNING: 'vehicle_description' is still present. Drop it before D.1 one-hot encoding.")
    else:
        print("\nOK: 'vehicle_description' is not present (safe for one-hot encoding).")

# Run checks
check_df("training_df_eng", training_df_eng)
check_df("validation_df_eng", validation_df_eng)
check_df("testing_df_eng", testing_df_eng)


SECTION C HEALTH CHECK: training_df_eng
Engineered columns present: ['vehicle_age', 'km_per_year', 'desc_len', 'engine_power_index', 'fuel_efficiency_score', 'brand_tier', 'state', 'location_coarse', 'is_new_vehicle']


,dtype,missing_pct,n_unique
vehicle_age,float64,0.0,38
km_per_year,float64,0.0,8618
desc_len,int64,0.0,76
engine_power_index,float64,0.0,9
fuel_efficiency_score,float64,0.0,143
brand_tier,object,0.0,3
state,object,0.0,6
location_coarse,object,0.0,31
is_new_vehicle,int64,0.0,2



vehicle_age: min=9.0 max=67.0 median=12.0

km_per_year: min=0.1 max=37761.75 median=10369.18125

desc_len: min=11 max=90 median=27.0

engine_power_index: min=0.0 max=144.0 median=16.0

fuel_efficiency_score: min=0.037037037037037035 max=0.5882352941176471 median=0.125

is_new_vehicle: min=0 max=1 median=0.0

OK: 'vehicle_description' is not present (safe for one-hot encoding).

SECTION C HEALTH CHECK: validation_df_eng
Engineered columns present: ['vehicle_age', 'km_per_year', 'desc_len', 'engine_power_index', 'fuel_efficiency_score', 'brand_tier', 'state', 'location_coarse', 'is_new_vehicle']


,dtype,missing_pct,n_unique
vehicle_age,float64,0.0,2
km_per_year,float64,0.0,2581
desc_len,int64,0.0,58
engine_power_index,float64,0.0,8
fuel_efficiency_score,float64,0.0,89
brand_tier,object,0.0,3
state,object,0.0,6
location_coarse,object,0.0,29
is_new_vehicle,int64,0.0,2



vehicle_age: min=6.0 max=7.0 median=7.0

km_per_year: min=2.0 max=49589.71428571428 median=8614.833333333334

desc_len: min=12 max=77 median=30.0

engine_power_index: min=0.0 max=144.0 median=16.0

fuel_efficiency_score: min=0.05952380952380952 max=0.4 median=0.13513513513513511

is_new_vehicle: min=0 max=1 median=0.0

OK: 'vehicle_description' is not present (safe for one-hot encoding).

SECTION C HEALTH CHECK: testing_df_eng
Engineered columns present: ['vehicle_age', 'km_per_year', 'desc_len', 'engine_power_index', 'fuel_efficiency_score', 'brand_tier', 'state', 'location_coarse', 'is_new_vehicle']


,dtype,missing_pct,n_unique
vehicle_age,float64,0.0,2
km_per_year,float64,0.0,1130
desc_len,int64,0.0,55
engine_power_index,float64,0.0,8
fuel_efficiency_score,float64,0.0,103
brand_tier,object,0.0,3
state,object,0.0,6
location_coarse,object,0.0,24
is_new_vehicle,int64,0.0,2



vehicle_age: min=3.0 max=4.0 median=4.0

km_per_year: min=0.25 max=44850.333333333336 median=696.0

desc_len: min=11 max=79 median=30.0

engine_power_index: min=0.0 max=144.0 median=16.0

fuel_efficiency_score: min=0.06666666666666667 max=0.625 median=0.12658227848101264

is_new_vehicle: min=0 max=1 median=0.0

OK: 'vehicle_description' is not present (safe for one-hot encoding).


In [ ]:
import numpy as np
import pandas as pd

cols = ["vehicle_age","km_per_year","desc_len","engine_power_index","fuel_efficiency_score","brand_tier","state","location_coarse","is_new_vehicle"]
cols = [c for c in cols if c in training_df_eng.columns]

summary = pd.DataFrame({
    "dtype": training_df_eng[cols].dtypes.astype(str),
    "missing_pct_train": (training_df_eng[cols].isna().mean()*100).round(2),
    "min_train": [training_df_eng[c].replace([np.inf, -np.inf], np.nan).min() if pd.api.types.is_numeric_dtype(training_df_eng[c]) else None for c in cols],
    "max_train": [training_df_eng[c].replace([np.inf, -np.inf], np.nan).max() if pd.api.types.is_numeric_dtype(training_df_eng[c]) else None for c in cols],
    "n_unique_train": [training_df_eng[c].nunique(dropna=True) for c in cols],
})
display(summary)

# Check for inf values in numeric engineered columns
num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(training_df_eng[c])]
inf_counts = {c: np.isinf(training_df_eng[c]).sum() for c in num_cols}
print("Inf counts (should be 0):", inf_counts)

# Check raw text presence
print("vehicle_description present?", "vehicle_description" in training_df_eng.columns)

,dtype,missing_pct_train,min_train,max_train,n_unique_train
vehicle_age,float64,0.0,9.000000,67.000000,38
km_per_year,float64,0.0,0.100000,37761.750000,8618
desc_len,int64,0.0,11.000000,90.000000,76
engine_power_index,float64,0.0,0.000000,144.000000,9
fuel_efficiency_score,float64,0.0,0.037037,0.588235,143
brand_tier,object,0.0,NaN,NaN,3
state,object,0.0,NaN,NaN,6
location_coarse,object,0.0,NaN,NaN,31
is_new_vehicle,int64,0.0,0.000000,1.000000,2


Inf counts (should be 0): {'vehicle_age': np.int64(0), 'km_per_year': np.int64(0), 'desc_len': np.int64(0), 'engine_power_index': np.int64(0), 'fuel_efficiency_score': np.int64(0), 'is_new_vehicle': np.int64(0)}
vehicle_description present? False


In [ ]:
cols = ["vehicle_age","km_per_year","desc_len","engine_power_index","fuel_efficiency_score","brand_tier","state","location_coarse","is_new_vehicle"]
for c in cols:
    if c in training_df_eng.columns:
        print("\n", c)
        print(" train missing %:", round(training_df_eng[c].isna().mean()*100, 2))
        print("   val missing %:", round(validation_df_eng[c].isna().mean()*100, 2))
        print("  test missing %:", round(testing_df_eng[c].isna().mean()*100, 2))


 vehicle_age
 train missing %: 0.0
   val missing %: 0.0
  test missing %: 0.0

 km_per_year
 train missing %: 0.0
   val missing %: 0.0
  test missing %: 0.0

 desc_len
 train missing %: 0.0
   val missing %: 0.0
  test missing %: 0.0

 engine_power_index
 train missing %: 0.0
   val missing %: 0.0
  test missing %: 0.0

 fuel_efficiency_score
 train missing %: 0.0
   val missing %: 0.0
  test missing %: 0.0

 brand_tier
 train missing %: 0.0
   val missing %: 0.0
  test missing %: 0.0

 state
 train missing %: 0.0
   val missing %: 0.0
  test missing %: 0.0

 location_coarse
 train missing %: 0.0
   val missing %: 0.0
  test missing %: 0.0

 is_new_vehicle
 train missing %: 0.0
   val missing %: 0.0
  test missing %: 0.0


In [ ]:
(training_df_eng["engine_power_index"] == 0).sum(), training_df_eng["engine_power_index"].describe()

(np.int64(5),
 count    8774.000000
 mean       21.703328
 std        12.486221
 min         0.000000
 25%        16.000000
 50%        16.000000
 75%        16.000000
 max       144.000000
 Name: engine_power_index, dtype: float64)

In [ ]:
cap = training_df_eng["km_per_year"].quantile(0.99)

for df in [training_df_eng, validation_df_eng, testing_df_eng]:
    df["km_per_year"] = df["km_per_year"].clip(upper=cap)

print("Capped km_per_year at 99th percentile:", cap)

Capped km_per_year at 99th percentile: 25575.90686868688


---
## D. Data Preparation for Modeling

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()

  y_train = X_train.pop(target_name)
  y_val = X_val.pop(target_name)
  y_test = X_test.pop(target_name)
except Exception as e:
  print(e)

### D.1 Data Transformation :- One-hot encode categoricals + align train/val/test columns


In [ ]:
# =========================
# D.1 Data Transformation (UPDATED):
# One-hot encode categoricals + align train/val/test columns
# =========================
import numpy as np
import pandas as pd

target_name = "price"

# 1) Split X/y
X_train = training_df_eng.drop(columns=[target_name]).copy()
y_train = training_df_eng[target_name].copy()

X_val = validation_df_eng.drop(columns=[target_name]).copy()
y_val = validation_df_eng[target_name].copy()

X_test = testing_df_eng.drop(columns=[target_name]).copy()
y_test = testing_df_eng[target_name].copy()

# 2) Ensure all object columns are strings (safe for get_dummies)
for df in [X_train, X_val, X_test]:
    obj_cols = df.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        df[c] = df[c].astype(str).fillna("unknown")

# 3) One-hot encode based on TRAIN categories, then align val/test
X_train_enc = pd.get_dummies(X_train, drop_first=False)
X_val_enc = pd.get_dummies(X_val, drop_first=False)
X_test_enc = pd.get_dummies(X_test, drop_first=False)

# Align columns to training (missing columns filled with 0)
X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print("D.1 complete.")
print("X_train_enc shape:", X_train_enc.shape)
print("X_val_enc shape:  ", X_val_enc.shape)
print("X_test_enc shape: ", X_test_enc.shape)

# Quick check: same columns?
assert list(X_train_enc.columns) == list(X_val_enc.columns) == list(X_test_enc.columns)

# Optional: check for any remaining NaNs (should be none)
print("\nNaNs in X_train_enc:", X_train_enc.isna().sum().sum())
print("NaNs in X_val_enc:  ", X_val_enc.isna().sum().sum())
print("NaNs in X_test_enc: ", X_test_enc.isna().sum().sum())

D.1 complete.
X_train_enc shape: (8774, 502)
X_val_enc shape:   (2615, 502)
X_test_enc shape:  (2647, 502)

NaNs in X_train_enc: 0
NaNs in X_val_enc:   0
NaNs in X_test_enc:  0


In [ ]:
data_transformation_1_explanations = """
Transformation: One-hot encoding of categorical variables (fit on training categories) and column alignment across validation/testing.
Why:
- Most ML models require numeric inputs.
- Validation/testing may contain unseen categories; alignment ensures consistent feature columns.
How:
- One-hot encoded categorical columns.
- Reindexed validation/testing to training columns with fill_value=0 (missing dummy columns become 0).
Impact:
- Produces model-ready X_train/X_val/X_test with identical column structure.
"""
print(data_transformation_1_explanations)


Transformation: One-hot encoding of categorical variables (fit on training categories) and column alignment across validation/testing.
Why:
- Most ML models require numeric inputs.
- Validation/testing may contain unseen categories; alignment ensures consistent feature columns.
How:
- One-hot encoded categorical columns.
- Reindexed validation/testing to training columns with fill_value=0 (missing dummy columns become 0).
Impact:
- Produces model-ready X_train/X_val/X_test with identical column structure.



In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### D.2 Data Transformation <put_name_here>

In [ ]:
# =========================
# D.2 (Optional but recommended for linear models):
# Standardize numeric features AFTER one-hot encoding
# (Do NOT standardize y)
# =========================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=False)  # with_mean=False is safe for sparse-like dummy matrices

X_train_scaled = scaler.fit_transform(X_train_enc)
X_val_scaled = scaler.transform(X_val_enc)
X_test_scaled = scaler.transform(X_test_enc)

print("Scaling complete.")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:  ", X_val_scaled.shape)
print("X_test_scaled shape: ", X_test_scaled.shape)

Scaling complete.
X_train_scaled shape: (8774, 502)
X_val_scaled shape:   (2615, 502)
X_test_scaled shape:  (2647, 502)


In [ ]:
# <Student to fill this section and then remove this comment>
data_transformation_2_explanations = """
Optional transformation (not applied here): feature scaling (StandardScaler/RobustScaler).
- Needed mainly for distance-based or linear models (e.g., KNN, linear regression, SVM).
- Not required for tree-based models (RandomForest, XGBoost).
- Recommended to apply scaling using a sklearn Pipeline during modeling to avoid leakage.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### D.3 Data Transformation 


In [ ]:
# D.x Target transform (optional)
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_test_log = np.log1p(y_test)

print("Using log1p(price) as target.")

Using log1p(price) as target.


In [ ]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler(with_centering=False)  # safe for dummy matrix style
X_train_rs = robust_scaler.fit_transform(X_train_enc)
X_val_rs = robust_scaler.transform(X_val_enc)
X_test_rs = robust_scaler.transform(X_test_enc)

In [ ]:
def add_frequency_encoding(train_df, other_df, col, new_col):
    freq = train_df[col].value_counts(normalize=True)
    return other_df[col].map(freq).fillna(0.0).rename(new_col)

# Example columns (adjust to what you have)
for col in ["model_name", "location_coarse"]:
    if col in X_train.columns:
        X_train[col + "_freq"] = add_frequency_encoding(X_train, X_train, col, col + "_freq")
        X_val[col + "_freq"] = add_frequency_encoding(X_train, X_val, col, col + "_freq")
        X_test[col + "_freq"] = add_frequency_encoding(X_train, X_test, col, col + "_freq")

print("Added frequency encoding columns.")

Added frequency encoding columns.


In [ ]:
# <Student to fill this section and then remove this comment>
data_transformation_3_explanations = """
Provide some explanations on why you believe it is important to perform this data transformation and its impacts
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

### D.n Fixing "outliers with clipping (winsorization) on numeric features"

> You can add more cells related to data preparation in this section

In [ ]:
clip_cols = ["kilometres_driven", "km_per_year", "fuel_consumption", "engine_power_index"]
clip_cols = [c for c in clip_cols if c in X_train.columns]

caps = {c: X_train[c].quantile(0.99) for c in clip_cols}

for df in [X_train, X_val, X_test]:
    for c, cap in caps.items():
        df[c] = df[c].clip(upper=cap)

print("Clipped numeric outliers at 99th percentile (train-based).")

Clipped numeric outliers at 99th percentile (train-based).


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# separate numeric vs categorical
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

print("Pipeline preprocessor ready. Use inside your model pipeline. ColumnTransformer + Pipeline (best practice)")

Pipeline preprocessor ready. Use inside your model pipeline. ColumnTransformer + Pipeline (best practice)


In [ ]:
# <Student to fill this section and then remove this comment>
data_transformation_n_explanations = """
Provide some explanations on why you believe it is important to perform this data transformation and its impacts
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_n_explanations', value=data_transformation_n_explanations)

---
## E. Save Datasets

> Do not change this code

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
  y_train.to_csv(at.folder_path / 'y_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)
  y_val.to_csv(at.folder_path / 'y_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
  y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fbbefce8-41ae-47c6-bc64-96decd566c0b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>